# 02 — Statistical Baseline (per ADI–CV² quadrant)

Classical per-pair forecasts on the same per-nest test windows used by the ML notebooks.

**Routing rule:**
| `demand_class` | Method |
|---|---|
| `Smooth` / `Erratic` | ETS (additive trend, no seasonality) — fit once on pre-test history, multi-step forecast |
| `Intermittent` | Croston (α tuned on the training tail) |
| `Lumpy` | SBA — bias-corrected Croston |

**Inputs:** `panel_weekly.csv`.
**Output:** `data/stat_predictions.csv` — one row per (modellable pair × test week).

Note: for Croston/SBA the recursion is computed once on the full series. By construction `fc[t]` only uses data up to `t-1`, so this is mathematically equivalent to a rolling one-step-ahead refit and avoids redundant work.


## Step 0 — Imports and load the weekly panel

In [2]:
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)

PAIR_COLS = ['NEST_NAME', 'HEALTH_FACILITY_NAME', 'PRODUCT_NAME']

file_path = ""
for root, dirs, files_in_dir in os.walk('/content'):
    if 'panel_weekly.parquet' in files_in_dir:
        file_path = os.path.join(root, 'panel_weekly.parquet')
        break

if not file_path:
    print("panel_weekly.parquet not found. Please upload it:")
    from google.colab import files
    uploaded = files.upload()
    if 'panel_weekly.parquet' in uploaded:
        file_path = os.path.join('/content', 'panel_weekly.parquet')

if file_path:
    DATA_DIR = Path(file_path).parent
    panel = pd.read_parquet(file_path)
    panel = panel.sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)
    mod = panel[panel['tier'] == 'modellable'].copy()

    print('Panel rows :', len(panel), '   Modellable rows:', len(mod))
    print('\nModellable pairs by demand_class:')
    print(mod[PAIR_COLS + ['demand_class']].drop_duplicates()['demand_class'].value_counts())
else:
    raise FileNotFoundError("panel_weekly.parquet is missing. Please upload the file.")

panel_weekly.parquet not found. Please upload it:


Saving panel_weekly.parquet to panel_weekly.parquet
Panel rows : 4091010    Modellable rows: 687678

Modellable pairs by demand_class:
demand_class
Intermittent    6558
Lumpy           1627
Smooth            95
Erratic           10
Name: count, dtype: int64


## Step 1 — Croston / SBA forecast functions

These two are the only `def`s we keep — they are pure math formulae and are clearer this way than inline.

In [3]:
def croston_forecast(y, alpha=0.1):
    """Classic Croston: smooth non-zero level z and inter-arrival mean p; forecast = z/p.
    Returns an array of length len(y); fc[t] uses only y[:t] (no peek at y[t])."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    fc = np.full(n, np.nan)
    if (y > 0).sum() == 0:
        return fc
    first_nz = int(np.argmax(y > 0))
    z = y[first_nz]
    p = float(first_nz + 1)
    q = 0
    for t in range(first_nz + 1, n):
        q += 1
        fc[t] = z / p
        if y[t] > 0:
            z = alpha * y[t] + (1 - alpha) * z
            p = alpha * q + (1 - alpha) * p
            q = 0
    return fc


def sba_forecast(y, alpha=0.1):
    """Syntetos-Boylan: (1 - alpha/2) * Croston, removes Croston's positive bias."""
    return croston_forecast(y, alpha=alpha) * (1 - alpha / 2)


## Step 2 — Per-pair α tuning helper (small grid on the training tail)

In [4]:
def pick_alpha(y_train, forecaster, alphas=(0.05, 0.1, 0.2, 0.3)):
    y_train = np.asarray(y_train, dtype=float)
    if len(y_train) < 8:
        return 0.1
    split = max(4, len(y_train) - 8)
    best_a, best_mae = 0.1, np.inf
    for a in alphas:
        fc = forecaster(y_train, a)
        err = np.abs(fc[split:] - y_train[split:])
        mae = float(np.nanmean(err)) if np.any(np.isfinite(err)) else np.inf
        if np.isfinite(mae) and mae < best_mae:
            best_a, best_mae = a, mae
    return best_a


## Step 3 — Main prediction loop (per modellable pair)

For each pair we:

1. Slice the pair's full series in chronological order.
2. Identify pre-test history (= train + valid rows) and the indexes of the 5 test rows.
3. **Croston / Lumpy**: run the recursion once on the FULL series — `fc[t]` uses only data before index `t` so we read off the forecast for each test week directly. Tune α on pre-test history.
4. **Smooth / Erratic**: fit ETS once on pre-test history and emit a 5-step-ahead forecast for the 5 test weeks; if ETS fails, fall back to last-value naïve.
5. Emit one row per test week with `(y_actual, y_pred, demand_class, method)`.


In [5]:
rows = []
n_pairs = mod[PAIR_COLS].drop_duplicates().shape[0]
print(f'Predicting {n_pairs:,} modellable pairs ...')

for i, ((nest, fac, prod), g) in enumerate(mod.groupby(PAIR_COLS, sort=False), start=1):
    if i % 1000 == 0 or i == n_pairs:
        print(f'  ... {i:,}/{n_pairs:,}')
    g = g.sort_values('week_start').reset_index(drop=True)
    y = g['weekly_units'].to_numpy()
    splits = g['split'].to_numpy()
    weeks = g['week_start'].to_numpy()
    dc = g['demand_class'].iloc[0]

    test_idx = np.where(splits == 'test')[0]
    if len(test_idx) == 0 or test_idx[0] == 0:
        continue
    pre_test_end = int(test_idx[0])
    y_pre = y[:pre_test_end]

    # Generate the test-window predictions per method.
    if dc in ('Intermittent', 'Lumpy'):
        forecaster = croston_forecast if dc == 'Intermittent' else sba_forecast
        alpha = pick_alpha(y_pre, forecaster)
        fc_full = forecaster(y, alpha=alpha)
        for t in test_idx:
            pred = float(fc_full[t]) if np.isfinite(fc_full[t]) else float(np.nanmean(y_pre)) if np.any(y_pre > 0) else 0.0
            rows.append({'NEST_NAME': nest, 'HEALTH_FACILITY_NAME': fac, 'PRODUCT_NAME': prod,
                         'week_start': weeks[t], 'y_actual': float(y[t]),
                         'y_pred': max(0.0, pred),
                         'demand_class': dc, 'method': 'Croston' if dc == 'Intermittent' else 'SBA'})
    elif dc in ('Smooth', 'Erratic'):
        try:
            fit = ExponentialSmoothing(np.maximum(y_pre, 0), trend='add', seasonal=None,
                                       initialization_method='estimated').fit()
            preds = np.maximum(fit.forecast(len(test_idx)), 0)
            method = 'ETS'
        except Exception:
            preds = np.full(len(test_idx), float(y_pre[-1]))
            method = 'naive_last'
        for k, t in enumerate(test_idx):
            rows.append({'NEST_NAME': nest, 'HEALTH_FACILITY_NAME': fac, 'PRODUCT_NAME': prod,
                         'week_start': weeks[t], 'y_actual': float(y[t]),
                         'y_pred': float(preds[k]),
                         'demand_class': dc, 'method': method})

stat_pred = pd.DataFrame(rows)
print('\nPrediction rows :', len(stat_pred))
print('Method counts   :')
print(stat_pred['method'].value_counts())


Predicting 8,290 modellable pairs ...
  ... 1,000/8,290
  ... 2,000/8,290
  ... 3,000/8,290
  ... 4,000/8,290
  ... 5,000/8,290
  ... 6,000/8,290
  ... 7,000/8,290
  ... 8,000/8,290
  ... 8,290/8,290

Prediction rows : 41450
Method counts   :
method
Croston    32790
SBA         8135
ETS          525
Name: count, dtype: int64


## Step 4 — Snapshot (per nest × per demand class) and save

In [6]:
def _wape(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.sum(np.abs(p - y)) / max(np.sum(np.abs(y)), 1e-9))

def _mae(y, p):
    return float(np.mean(np.abs(np.asarray(p, dtype=float) - np.asarray(y, dtype=float))))

snap = (stat_pred.groupby(['NEST_NAME', 'demand_class'])
                  .apply(lambda g: pd.Series({'n': len(g),
                                              'MAE':  _mae(g['y_actual'], g['y_pred']),
                                              'WAPE': _wape(g['y_actual'], g['y_pred'])}))
                  .reset_index())
print('Per nest x demand_class snapshot:')
print(snap.to_string(index=False))

out = DATA_DIR / 'stat_predictions.csv'
stat_pred.to_csv(out, index=False)
print(f'\nSaved {out}  size={out.stat().st_size/1e6:.2f} MB')


Per nest x demand_class snapshot:
  NEST_NAME demand_class       n       MAE     WAPE
GH1 Omenako Intermittent  3950.0  1.571666 1.474253
GH1 Omenako        Lumpy   290.0  2.457398 1.587184
  GH3 Vobsi Intermittent  5445.0  1.675945 1.511599
  GH3 Vobsi        Lumpy  4085.0  4.497743 1.292346
RW1 Muhanga      Erratic    40.0  1.892515 0.694501
RW1 Muhanga Intermittent 14960.0  6.468887 1.732541
RW1 Muhanga        Lumpy  2540.0 11.201006 1.664457
RW1 Muhanga       Smooth   230.0  1.743302 0.679592
RW2 Kayonza      Erratic    10.0  1.572690 0.786345
RW2 Kayonza Intermittent  8435.0  6.204695 1.756203
RW2 Kayonza        Lumpy  1220.0 16.268265 1.423765
RW2 Kayonza       Smooth   245.0  1.714352 0.623170

Saved /content/stat_predictions.csv  size=4.53 MB
